# Signal Engineering\n\nThis notebook demonstrates a simple signal engineering workflow using synthetic price data.\n\n## Goals\n- Create a mock price series\n- Engineer rolling features\n- Build a z-score mean-reversion style signal\n- Visualize signal behavior and next-period returns

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nnp.random.seed(42)

In [ ]:
# Generate a synthetic price series with mild drift and noise\nn = 500\ndates = pd.date_range('2023-01-01', periods=n, freq='B')\nreturns = np.random.normal(loc=0.0003, scale=0.01, size=n)\nprice = 100 * (1 + pd.Series(returns, index=dates)).cumprod()\n\ndf = pd.DataFrame({'price': price})\ndf['ret_1d'] = df['price'].pct_change()\ndf.head()

In [ ]:
# Rolling features\nlookback = 20\ndf['roll_mean_20'] = df['price'].rolling(lookback).mean()\ndf['roll_std_20'] = df['price'].rolling(lookback).std()\ndf['zscore_20'] = (df['price'] - df['roll_mean_20']) / df['roll_std_20']\n\n# Next-day return target (for simple inspection)\ndf['fwd_ret_1d'] = df['ret_1d'].shift(-1)

In [ ]:
# Mean-reversion style signal from z-score\n# Long when price is sufficiently below rolling mean, short when above\nthreshold = 1.0\ndf['signal'] = 0\ndf.loc[df['zscore_20'] <= -threshold, 'signal'] = 1\ndf.loc[df['zscore_20'] >= threshold, 'signal'] = -1\n\n# Shift signal to avoid look-ahead bias (enter next bar)\ndf['signal_shifted'] = df['signal'].shift(1).fillna(0)\n\n# Signal-aligned simple returns\ndf['signal_pnl_1d'] = df['signal_shifted'] * df['ret_1d']\ndf[['price', 'zscore_20', 'signal', 'signal_shifted', 'signal_pnl_1d']].tail()

In [ ]:
# Plot price and rolling mean\nplt.figure(figsize=(12, 5))\nplt.plot(df.index, df['price'], label='Price')\nplt.plot(df.index, df['roll_mean_20'], label='20D Rolling Mean')\nplt.title('Synthetic Price Series with Rolling Mean')\nplt.xlabel('Date')\nplt.ylabel('Price')\nplt.legend()\nplt.tight_layout()\nplt.show()

In [ ]:
# Plot z-score and thresholds\nplt.figure(figsize=(12, 4))\nplt.plot(df.index, df['zscore_20'], label='Z-Score (20D)')\nplt.axhline(1.0, linestyle='--')\nplt.axhline(-1.0, linestyle='--')\nplt.axhline(0.0, linestyle=':')\nplt.title('Signal Feature: Rolling Z-Score')\nplt.xlabel('Date')\nplt.ylabel('Z-Score')\nplt.legend()\nplt.tight_layout()\nplt.show()

In [ ]:
# Compare cumulative returns: buy-and-hold vs signal strategy (toy example)\ndf_plot = df.dropna().copy()\ndf_plot['cum_buy_hold'] = (1 + df_plot['ret_1d']).cumprod()\ndf_plot['cum_signal'] = (1 + df_plot['signal_pnl_1d']).cumprod()\n\nplt.figure(figsize=(12, 5))\nplt.plot(df_plot.index, df_plot['cum_buy_hold'], label='Buy & Hold')\nplt.plot(df_plot.index, df_plot['cum_signal'], label='Signal Strategy (Toy)')\nplt.title('Cumulative Return Comparison (Illustrative)')\nplt.xlabel('Date')\nplt.ylabel('Growth of $1')\nplt.legend()\nplt.tight_layout()\nplt.show()

## Notes\n\n- This is a **toy signal engineering example** using synthetic data for demonstration.\n- The focus is on workflow and feature construction, not production alpha claims.\n- In a live research setting, this would be extended with:\n  - transaction costs/slippage\n  - regime filters\n  - turnover constraints\n  - out-of-sample validation